In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('0.1_filtered_dataset.csv')
output_dir = 'Assets/'
os.makedirs(output_dir, exist_ok=True)

num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()

# HISTOGRAMS & KDE PLOTS
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(20, 16))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='#4C72B0', bins=25, edgecolor='black', alpha=0.7)
    axes[i].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

hist_img_name = '0.3_hist_kde_plot_datacut_pre_model.png'
hist_img_path = os.path.join(output_dir, hist_img_name)
plt.savefig(hist_img_path, dpi=300)
print(f"![Histograms & KDE]({hist_img_path})")
plt.close()

# DATATYPES

print("\n DATATYPES")
print(df.dtypes.to_string())

# MISSING VALUES & MISSING DATA PLOT
print("\n MISSING VALUES")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
col_types = df.dtypes

miss_df = pd.DataFrame({
    'Missing Count': missing, 
    'Missing %': missing_pct,
    'Data Type': col_types
})
miss_df = miss_df[miss_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

if not miss_df.empty:
    print(miss_df.to_string())
print(f"\n Total missing cells: {missing.sum()}")
print(f"\n Features with missing values: {(missing > 0).sum()}/{len(df.columns)}")

if not miss_df.empty:
    plt.figure(figsize=(12, 6))
    ax = sns.barplot(x=miss_df.index, y='Missing %', data=miss_df, palette='flare')
    for p in ax.patches:
        ax.annotate(f'{p.get_height()}%',
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='bottom',
                    fontsize=10, fontweight='bold', color='black', xytext=(0, 5), textcoords='offset points')
    plt.title('Percentage of Missing Data by Feature', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Missing Percentage (%)', fontsize=12)
    plt.xlabel('Features', fontsize=12)
    plt.ylim(0, max(miss_df['Missing %']) + 5) 
    plt.tight_layout()
    
    miss_img_name = '0.3_missing_data_plot_datacut_pre_model.png'
    miss_img_path = os.path.join(output_dir, miss_img_name)
    plt.savefig(miss_img_path, dpi=300)
    print(f"\n![Missing Data Plot]({miss_img_path})")
    plt.close()
else:
    print("\nGreat! The dataset has no missing values.")

# OUTLIER DETECTION (IQR METHOD)
print("\n OUTLIER DETECTION (IQR Method)")
outlier_rows = set()
for col in num_cols:
    s = df[col].dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lb, ub = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    mask = (df[col] < lb) | (df[col] > ub)
    n_out = mask.sum()
    outlier_rows.update(df[mask].index.tolist())
    print(f"  {col:<28} | Outliers: {n_out} ({n_out/len(s)*100:.2f}%) | Range: [{lb:.1f}, {ub:.1f}]")

print(f"Total rows with at least 1 outlier: {len(outlier_rows)}")

# RELATIONSHIPS
print("\n RELATIONSHIP WITH STRESS LEVEL")
groupby_stress = df.groupby('Stress_Level')[['Work_Hours_Per_Week', 'Screen_Time_Hours', 'Sleep_Hours']].mean()
print(groupby_stress.round(2))

print("\n RELATIONSHIP WITH BURNOUT RISK")
groupby_burnout = df.groupby('Burnout_Risk')[['Work_Hours_Per_Week', 'Screen_Time_Hours', 'Sleep_Hours']].mean()
print(groupby_burnout.round(2))

# BURNOUT RISK TREND PLOTS
burnout_order = ['Low', 'Moderate', 'High']

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

sns.pointplot(data=df, x='Burnout_Risk', y='Work_Hours_Per_Week', order=burnout_order, 
              ax=axes[0], color='#d62728', markers="o", linestyles="-")
axes[0].set_title('Work Hours per Week vs Burnout Risk', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Average Work Hours (Week)', fontsize=12)
axes[0].set_xlabel('Burnout Risk', fontsize=12)

sns.pointplot(data=df, x='Burnout_Risk', y='Screen_Time_Hours', order=burnout_order, 
              ax=axes[1], color='#ff7f0e', markers="s", linestyles="--")
axes[1].set_title('Screen Time vs Burnout Risk', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Average Screen Time (Hours/Day)', fontsize=12)
axes[1].set_xlabel('Burnout Risk', fontsize=12)

sns.pointplot(data=df, x='Burnout_Risk', y='Sleep_Hours', order=burnout_order, 
              ax=axes[2], color='#2ca02c', markers="D", linestyles="-.")
axes[2].set_title('Sleep Hours vs Burnout Risk', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Average Sleep Hours (Day)', fontsize=12)
axes[2].set_xlabel('Burnout Risk', fontsize=12)

plt.suptitle('Trends of Key Variables Across Burnout Risk Levels', fontsize=18, fontweight='bold', y=1.05)
plt.tight_layout()

trend_img_name = '0.3_burnout_risk_trends.png'
trend_img_path = os.path.join(output_dir, trend_img_name)
plt.savefig(trend_img_path, dpi=300, bbox_inches='tight')
print(f"\n![Trend Plot]({trend_img_path})")
plt.close()

![Histograms & KDE](Assets/0.3_hist_kde_plot_datacut_pre_model.png)

 DATATYPES
Age                        float64
Gender                         str
Employment_Status              str
Work_Hours_Per_Week          int64
Screen_Time_Hours          float64
Sleep_Hours                float64
Sleep_Quality                  str
Physical_Activity_Hours    float64
Meditation_Minutes         float64
Coffee_Cups_Per_Day          int64
Stress_Level                   str
Chronic_Stress                 str
Burnout_Risk                   str
Occupation                     str
Education_Level                str

 MISSING VALUES
                         Missing Count  Missing % Data Type
Meditation_Minutes                1200        2.4   float64
Physical_Activity_Hours           1100        2.2   float64
Sleep_Hours                        950        1.9   float64
Screen_Time_Hours                  850        1.7   float64
Age                                250        0.5   float64

 Total missing ce

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df_orig = pd.read_csv('0.1_filtered_dataset.csv')
df_proc = pd.read_csv('0.2_processed_data.csv')

output_dir = 'Assets/'
os.makedirs(output_dir, exist_ok=True)

imputed_cols = df_orig.columns[df_orig.isnull().any()].tolist()

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(imputed_cols):

    sns.histplot(df_orig[col].dropna(), color='#4C72B0', label='Before Fill (Original)', 
                 kde=True, stat="density", bins=30, alpha=0.4, ax=axes[i])
    

    sns.histplot(df_proc[col], color='#DD8452', label='After Fill (Mean)', 
                 kde=True, stat="density", bins=30, alpha=0.4, ax=axes[i])
    
    
    mean_val = df_proc[col].mean()
    axes[i].axvline(mean_val, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.2f}')
    
    axes[i].set_title(f'Bias Check: {col}', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Density')
    axes[i].legend()

for j in range(len(imputed_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Data Shift/Bias Before and After Mean Imputation', fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()

bias_img_name = '0.3_imputation_bias_check.png'
bias_img_path = os.path.join(output_dir, bias_img_name)
plt.savefig(bias_img_path, dpi=300, bbox_inches='tight')
print(f"\n![Bias Check Plot]({bias_img_path})")
plt.close()


![Bias Check Plot](/Users/hungdan/Documents/pj_02/early-burnout-risk-prediction/Assets/0.3_imputation_bias_check.png)
